# Data Exploration Snapshot

This notebook profiles **all available monthly parquet files** to support presentation slides (combined size, schema checks, and a sample preview).

In [6]:

from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Resolve project root from common notebook working directories.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
project_root = next((p for p in candidate_roots if (p / "README.md").exists() and (p / "src").exists()), cwd)

# Collect all monthly parquet files from common project locations.
data_dirs = [project_root / "data" / "raw", project_root / "data"]
parquet_files = []
for d in data_dirs:
    if d.exists():
        parquet_files.extend(sorted(d.glob("data-*.parquet")))

# De-duplicate files while preserving order.
seen = set()
unique_files = []
for f in parquet_files:
    r = f.resolve()
    if r not in seen:
        seen.add(r)
        unique_files.append(f)

if not unique_files:
    # Fallback: search recursively under data/ in case files are nested differently.
    unique_files = sorted((project_root / "data").rglob("data-*.parquet"))

if not unique_files:
    raise FileNotFoundError(
        f"No monthly parquet files found under {project_root / 'data'} (including subfolders)."
    )

print(f"✅ Project root: {project_root}")
print(f"✅ Found {len(unique_files)} parquet file(s)")
for f in unique_files:
    print(f"  - {f.relative_to(project_root)}")

# Open metadata for each file without loading full datasets into RAM.
meta_rows = 0
schema_reference = None
for f in unique_files:
    pf = pq.ParquetFile(f)
    meta_rows += pf.metadata.num_rows
    if schema_reference is None:
        schema_reference = pf.schema.names

print(f"Total rows across all files: {meta_rows:,}")
print(f"Columns in schema: {len(schema_reference)}")
print("--- Column names ---")
print(schema_reference)

✅ Project root: C:\Users\Gangadhar\OneDrive\Study\EBS\Hackathon\Latest\Business-Analytics-Hackathon
✅ Found 18 parquet file(s)
  - data\data-2024-07.parquet
  - data\data-2024-08.parquet
  - data\data-2024-09.parquet
  - data\data-2024-10.parquet
  - data\data-2024-11.parquet
  - data\data-2024-12.parquet
  - data\data-2025-01.parquet
  - data\data-2025-02.parquet
  - data\data-2025-03.parquet
  - data\data-2025-04.parquet
  - data\data-2025-05.parquet
  - data\data-2025-06.parquet
  - data\data-2025-07.parquet
  - data\data-2025-08.parquet
  - data\data-2025-09.parquet
  - data\data-2025-10.parquet
  - data\data-2025-11.parquet
  - data\data-2025-12.parquet
Total rows across all files: 60,701,100
Columns in schema: 16
--- Column names ---
['station_name', 'xml_station_name', 'eva', 'train_name', 'final_destination_station', 'delay_in_min', 'time', 'is_canceled', 'train_type', 'train_line_ride_id', 'train_line_station_num', 'arrival_planned_time', 'arrival_change_time', 'departure_plan

In [7]:
# Build a presentation-friendly sample from all files (small slice per file).
sample_parts = []
for file_path in unique_files:
    part = pq.read_table(file_path).slice(0, 5).to_pandas()
    part["source_file"] = file_path.name
    sample_parts.append(part)

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df.head(15)

,station_name,xml_station_name,eva,train_name,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id,source_file
0,Bielefeld Hbf,Bielefeld Hbf,08000036,RE 6,Bielefeld Hbf,-1,2024-07-01,False,RE,2444314634072177884,23,2024-07-01 00:01:00,2024-07-01 00:00:00,NaT,NaT,2444314634072177884-2406302048-23,data-2024-07.parquet
1,Hamburg Hbf,Hamburg Hbf (S-Bahn),08098549,S 5,Hamburg Elbgaustraße,0,2024-07-01,False,S,1695252692593883835,7,2024-06-30 23:59:00,2024-06-30 23:59:00,2024-07-01 00:00:00,2024-07-01,1695252692593883835-2406302341-7,data-2024-07.parquet
2,None,"Hauptbahnhof, Saarbrücken",0836075,STB 1,"Riegelsberg Süd, Riegelsberg",0,2024-07-01,False,STB,,9,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01 00:00:00,2024-07-01,-788955727001224364-2406302347-9,data-2024-07.parquet
3,Nürnberg Hbf,Nürnberg Hbf,08000284,Bus EV,Nürnberg Hbf,0,2024-07-01,False,Bus,,2,2024-07-01 00:00:00,2024-07-01 00:00:00,NaT,NaT,-2337914853331985387-2406302300-2,data-2024-07.parquet
4,Singen (Hohentwiel),Singen(Hohentwiel),08000073,SBB S6,Engen,0,2024-07-01,False,SBB,8357206333431226217,12,2024-06-30 23:56:00,2024-06-30 23:56:00,2024-07-01 00:00:00,2024-07-01,8357206333431226217-2406302323-12,data-2024-07.parquet
5,None,Karlsruhe Bahnhofsvorplatz,08079041,S 8,"Tullastraße/Alter Schlachthof, Karlsruhe",3,2024-08-01,False,S,3950767135730977091,42,2024-07-31 23:56:00,2024-07-31 23:59:00,2024-07-31 23:57:00,2024-08-01,3950767135730977091-2407312153-42,data-2024-08.parquet
6,Hannover Hbf,Hannover Hbf,08000152,S 8,Hannover Hbf,8,2024-08-01,False,S,5466041995123654783,2,2024-07-31 23:52:00,2024-08-01 00:00:00,NaT,NaT,5466041995123654783-2407312345-2,data-2024-08.parquet
7,Neuss Hbf,Neuss Hbf,08000274,S 11,Bergisch Gladbach,4,2024-08-01,False,S,2862161729195150146,13,2024-07-31 23:54:00,2024-07-31 23:59:00,2024-07-31 23:56:00,2024-08-01,2862161729195150146-2407312324-13,data-2024-08.parquet
8,Ulm Hbf,Ulm Hbf,08000170,RE 75,Ulm Hbf,11,2024-08-01,False,RE,3231093823922762694,12,2024-07-31 23:49:00,2024-08-01 00:00:00,NaT,NaT,3231093823922762694-2407312235-12,data-2024-08.parquet
9,Singen (Hohentwiel),Singen(Hohentwiel),08000073,IRE 3,Singen (Hohentwiel),27,2024-08-01,False,IRE,6413319551723959065,23,2024-07-31 23:33:00,2024-08-01 00:00:00,NaT,NaT,6413319551723959065-2407312152-23,data-2024-08.parquet
